# 02 Silver: scoped and cleaned data

Builds the analysis dataset from the bronze views created in `01_bronze.ipynb`
(the views live in the shared DuckDB database, so bronze does not need to be re-run).

The thesis scope is applied first, so all later cleaning runs only on the data that is analysed.

## Setup

In [1]:
from config import PLATFORM_NAMES, PLATFORMS, SCHEMA_BREAK, SILVER_ROOT, connect

import json

import pandas as pd

con = connect()
pd.set_option("display.max_rows", 100)

## 1. Apply thesis scope

Silver keeps only statements that fall into the thesis scope:

| Rule | Column | Reason |
|---|---|---|
| Germany is in the territorial scope | `territorial_scope` contains `"DE"` | Thesis focuses on moderation decisions affecting Germany |
| Content posted in 2025 | `year(content_date) = 2025` | The thesis studies moderation of content posted in 2025 |

Note on `territorial_scope`: X always names exactly one country, while TikTok almost always lists the whole EEA.
The DE rule therefore keeps Germany-only statements on X but mostly EEA-wide statements on TikTok. The column
`n_countries_in_scope` is added so that analysis can distinguish Germany-only (`1`) from multi-country statements.

Limitation: the raw data contains only statements **submitted** to the database in 2025 (`created_at`). Content posted
in 2025 but moderated or reported in 2026 is not included, so the last weeks of 2025 are likely undercounted.

A third rule (v1 window only) is added in section 4, once sections 2 and 3 have established why the data after 2025-07-01 is not comparable. The views of this section therefore still cover the whole year.

The scoped data is defined as views; it is written to parquet once, after the cleaning steps.

In [2]:
con.execute("CREATE SCHEMA IF NOT EXISTS silver")

SCOPE_FILTER = """
    list_contains(from_json(territorial_scope, '["VARCHAR"]'), 'DE')
    AND year(content_date) = 2025
"""

for p in PLATFORMS:
    con.execute(f"""
        CREATE OR REPLACE VIEW silver.scoped_{p} AS
        SELECT *, json_array_length(territorial_scope) AS n_countries_in_scope
        FROM bronze.{p}
        WHERE {SCOPE_FILTER}
    """)

con.sql("SELECT table_schema, table_name FROM information_schema.tables WHERE table_schema = 'silver'").df()

,table_schema,table_name
0,silver,analysis_tiktok
1,silver,analysis_x
2,silver,scoped_tiktok
3,silver,scoped_x
4,silver,sor


Rows remaining after each rule (applied in order). The raw data is grouped by the two scope columns first,
so each platform needs only one scan.

In [3]:
funnel = []
for p in PLATFORMS:
    has_de, in_2025, total = con.sql(f"""
        WITH g AS (
            SELECT territorial_scope, year(content_date) = 2025 AS in_2025, count(*) AS n
            FROM bronze.{p} GROUP BY ALL
        ), f AS (
            SELECT coalesce(list_contains(from_json(territorial_scope, '["VARCHAR"]'), 'DE'), false) AS has_de, in_2025, n
            FROM g
        )
        SELECT sum(n) FILTER (WHERE has_de),
               sum(n) FILTER (WHERE has_de AND coalesce(in_2025, false)),
               sum(n)
        FROM f
    """).fetchone()
    for step, n in [("bronze (all rows)", total), ("territorial_scope contains DE", has_de), ("content_date in 2025", in_2025)]:
        funnel.append({"platform": p, "step": step, "rows": int(n or 0)})

funnel = pd.DataFrame(funnel)
funnel["removed"] = -funnel.groupby("platform")["rows"].diff().fillna(0).astype("int64")
funnel["kept_of_bronze_%"] = (100 * funnel["rows"] / funnel.groupby("platform")["rows"].transform("first")).round(2)
funnel

,platform,step,rows,removed,kept_of_bronze_%
0,tiktok,bronze (all rows),1002729268,0,100.00
1,tiktok,territorial_scope contains DE,999079277,3649991,99.64
2,tiktok,content_date in 2025,979896444,19182833,97.72
3,x,bronze (all rows),670093,0,100.00
4,x,territorial_scope contains DE,183324,486769,27.36
5,x,content_date in 2025,183321,3,27.36


## 2. Schema break on 1 July 2025

On 2025-07-01 the European Commission introduced an updated submission schema, which splits the reference year into
two parts (API v1 before, v2 from that date). Rows of the scoped data per window, by `content_date`:

| Window | Schema |
|---|---|
| 2025-01-01 to 2025-06-30 | v1 |
| 2025-07-01 to 2025-12-31 | v2 |

In [4]:
SCHEMA_BREAK = "2025-07-01"

con.sql(" UNION ALL ".join(f"""
    SELECT '{p}' AS platform,
           count(*) FILTER (WHERE content_date < DATE '{SCHEMA_BREAK}') AS v1_jan_jun,
           count(*) FILTER (WHERE content_date >= DATE '{SCHEMA_BREAK}') AS v2_jul_dec,
           count(*) AS total
    FROM silver.scoped_{p}""" for p in PLATFORMS) + " ORDER BY platform").df()

,platform,v1_jan_jun,v2_jul_dec,total
0,tiktok,824534503,155361941,979896444
1,x,132191,51130,183321


## 3. Schema break diagnostics

Is the analysis comparable across the break? Two checks on the scoped data, per platform and window
(v1 = `content_date` before 2025-07-01, v2 = from that date).

### 3.1 Which columns are filled per window

Share of non-null values per column. A column that is filled in one window and empty in the other is a
structural break in the record format.

In [5]:
RAW_COLS = [c for c in con.sql("SELECT * FROM silver.scoped_x LIMIT 0").columns
            if c not in ("dump_date", "source_file", "n_countries_in_scope")]

fill = []
for p in PLATFORMS:
    parts = []
    for c in RAW_COLS:
        for v, cond in (("v1", "<"), ("v2", ">=")):
            parts.append(f"100.0 * count({c}) FILTER (WHERE content_date {cond} DATE '{SCHEMA_BREAK}') "
                         f"/ nullif(count(*) FILTER (WHERE content_date {cond} DATE '{SCHEMA_BREAK}'), 0) "
                         f'AS "{c}__{v}"')
    row = con.sql(f"SELECT {', '.join(parts)} FROM silver.scoped_{p}").df().iloc[0]
    for c in RAW_COLS:
        fill.append({"column": c, "platform": p,
                     "v1_%": row[f"{c}__v1"], "v2_%": row[f"{c}__v2"]})

fill = pd.DataFrame(fill).round(2)
fill_wide = fill.pivot(index="column", columns="platform", values=["v1_%", "v2_%"])
fill_wide.columns = [f"{p}_{v.replace('_%', '')}" for v, p in fill_wide.columns]
fill_wide = fill_wide[sorted(fill_wide.columns)].fillna(0)

empty_both = fill_wide[(fill_wide == 0).all(axis=1)].index.tolist()
print(f"Columns empty in both windows on both platforms ({len(empty_both)}):", ", ".join(empty_both))

fill_wide.drop(index=empty_both)

Columns empty in both windows on both platforms (8): account_type, content_language, decision_ground_reference_url, decision_monetary, decision_monetary_other, end_date_account_restriction, incompatible_content_illegal, source_identity


,tiktok_v1,tiktok_v2,x_v1,x_v2
column,,,,
application_date,100.00,100.00,100.00,100.00
automated_decision,100.00,100.00,100.00,100.00
automated_detection,100.00,100.00,100.00,100.00
category,100.00,100.00,100.00,100.00
category_addition,0.00,0.00,0.02,0.00
category_specification,0.00,0.00,99.97,0.00
category_specification_other,0.00,0.00,45.79,0.00
content_date,100.00,100.00,100.00,100.00
content_type,100.00,100.00,100.00,100.00


The fill rates above are rounded to two decimal places. Two of the eight attributes listed as empty in both windows on both platforms, `decision_monetary` and `end_date_account_restriction`, are populated in a small number of TikTok records, quantified in section 4.4; the remaining six are unpopulated in every record.

Columns whose fill rate changes by more than 50 percentage points between the windows:

In [6]:
changed = []
for p in PLATFORMS:
    d = fill.query("platform == @p").set_index("column")
    diff = (d["v2_%"].fillna(0) - d["v1_%"].fillna(0))
    for c, v in diff[diff.abs() > 50].items():
        changed.append({"platform": p, "column": c,
                        "v1_%": d.loc[c, "v1_%"], "v2_%": d.loc[c, "v2_%"], "change_pp": round(v, 2)})

changed = pd.DataFrame(changed)
print(f"{len(changed)} column/platform combinations change by more than 50 pp")
changed

1 column/platform combinations change by more than 50 pp


,platform,column,v1_%,v2_%,change_pp
0,x,category_specification,99.97,0.0,-99.97


**Result.** The record format changes mainly on X: `category_specification` drops from 99.97% to 0% filled and
`category_specification_other` from 45.79% to 0%, i.e. X stops submitting the detailed classification from
2025-07-01. Smaller shifts (X `decision_account` 16.7 → 30.9%, TikTok `decision_visibility_other` 37.9 → 51.1%)
look like changed reporting practice rather than a changed format. Eight columns are empty in both windows on both
platforms.

Independent of the break, the platforms use different grounds: X always fills `illegal_content_*`,
TikTok always `incompatible_content_*`.

### 3.2 Which category values occur per window

Distinct values of the enum-like columns per window. Values that occur in only one window mean the taxonomy is not
comparable across the break. Columns holding JSON lists (e.g. `decision_visibility`) are unnested first, so
combinations are not counted as separate values.

In [7]:
ENUM_COLS = ["decision_visibility", "decision_monetary", "decision_provision", "decision_account",
             "account_type", "decision_ground", "category", "category_addition", "category_specification",
             "content_type", "source_type", "automated_detection", "automated_decision",
             "incompatible_content_illegal"]


def value_sets(platform: str) -> pd.DataFrame:
    """Distinct values per enum-like column and window, one scan per platform."""
    aggs = ", ".join(f'list(DISTINCT {c}) AS "{c}"' for c in ENUM_COLS)
    raw = con.sql(f"""
        SELECT content_date < DATE '{SCHEMA_BREAK}' AS is_v1, {aggs}
        FROM silver.scoped_{platform} GROUP BY is_v1
    """).df().set_index("is_v1")

    def flatten(values):
        """JSON list columns store e.g. '["A","B"]'; count single values, not combinations."""
        out = set()
        for v in [] if values is None else list(values):
            if v is None:
                continue
            v = str(v)
            out.update(json.loads(v) if v.startswith("[") else [v])
        return out

    rows = []
    for c in ENUM_COLS:
        v1, v2 = flatten(raw.loc[True, c]), flatten(raw.loc[False, c])
        rows.append({"platform": platform, "column": c,
                     "in_both": len(v1 & v2),
                     "only_v1": ", ".join(sorted(v1 - v2)) or "-",
                     "only_v2": ", ".join(sorted(v2 - v1)) or "-"})
    return pd.DataFrame(rows)


values = pd.concat([value_sets(p) for p in PLATFORMS], ignore_index=True)

for r in values.itertuples():
    if r.only_v1 == "-" and r.only_v2 == "-":
        continue
    print(f"{r.platform}.{r.column}: {r.in_both} values in both windows")
    for label, vals in (("only v1", r.only_v1), ("only v2", r.only_v2)):
        if vals != "-":
            print(f"   {label}:")
            for v in sorted(vals.split(", ")):
                print(f"      {v}")
    print()

print("Columns with an identical value set in both windows:")
values[(values["only_v1"] == "-") & (values["only_v2"] == "-")][["platform", "column", "in_both"]]


tiktok.decision_monetary: 0 values in both windows
   only v1:
      DECISION_MONETARY_SUSPENSION

tiktok.category: 15 values in both windows
   only v1:
      STATEMENT_CATEGORY_NON_CONSENSUAL_BEHAVIOUR
      STATEMENT_CATEGORY_UNSAFE_AND_ILLEGAL_PRODUCTS

x.category: 12 values in both windows
   only v1:
      STATEMENT_CATEGORY_NON_CONSENSUAL_BEHAVIOUR
      STATEMENT_CATEGORY_PORNOGRAPHY_OR_SEXUALIZED_CONTENT
      STATEMENT_CATEGORY_SCOPE_OF_PLATFORM_SERVICE
      STATEMENT_CATEGORY_UNSAFE_AND_ILLEGAL_PRODUCTS

x.category_addition: 0 values in both windows
   only v1:
      STATEMENT_CATEGORY_ILLEGAL_OR_HARMFUL_SPEECH
      STATEMENT_CATEGORY_NON_CONSENSUAL_BEHAVIOUR
      STATEMENT_CATEGORY_PORNOGRAPHY_OR_SEXUALIZED_CONTENT
      STATEMENT_CATEGORY_SELF_HARM
      STATEMENT_CATEGORY_VIOLENCE

x.category_specification: 0 values in both windows
   only v1:
      KEYWORD_ADULT_SEXUAL_MATERIAL
      KEYWORD_CHILD_SEXUAL_ABUSE_MATERIAL
      KEYWORD_COORDINATED_HARM
      KEYWORD_COPY

,platform,column,in_both
0,tiktok,decision_visibility,5
2,tiktok,decision_provision,1
3,tiktok,decision_account,2
4,tiktok,account_type,0
5,tiktok,decision_ground,2
7,tiktok,category_addition,0
8,tiktok,category_specification,0
9,tiktok,content_type,6
10,tiktok,source_type,4
11,tiktok,automated_detection,2


**Result.** Category values only ever disappear at the break, none are added. X loses 4 of 16 `category` values
(including `PORNOGRAPHY_OR_SEXUALIZED_CONTENT` and `NON_CONSENSUAL_BEHAVIOUR`), all 5 `category_addition` values and
all 16 `KEYWORD_*` values of `category_specification`. TikTok loses 2 of 17 `category` values and the only
`decision_monetary` value. All other enum-like columns keep an identical value set.

**Conclusion.** The classification available before 2025-07-01 is finer than afterwards, and on X a whole
classification level disappears. The two windows are therefore not comparable without harmonisation, which
justifies restricting the analysis to the v1 window (2025-01-01 to 2025-06-30). Check 3 (jump in the
distributions) is not needed: the structural evidence is unambiguous.

### 3.3 Decision: keep v2 or not

**What was checked.** Three questions, both platforms, on the scoped data, split at 2025-07-01:
1. *Does the record format change?* Share of non-null values per column and window (3.1).
2. *Does the classification change?* Distinct values of the enum-like columns per window (3.2).
3. *Does the volume change?* Rows per window (section 2).

**What was found.**

| Evidence | v1 | v2 |
|---|---|---|
| X: `category_specification` filled | 99.97% | 0% |
| X: `category_specification_other` filled | 45.79% | 0% |
| X: distinct `category` values | 16 | 12 |
| X: distinct `category_addition` values | 5 | 0 |
| TikTok: distinct `category` values | 17 | 15 |
| TikTok: rows (scope: DE, content posted 2025) | 824,534,503 | 155,361,941 |
| X: rows (same scope) | 132,191 | 51,130 |

Category values only ever disappear at the break; none are added. On X an entire classification level
(`category_specification`, the `KEYWORD_*` taxonomy) stops being reported.

**Why harmonisation does not solve it.**
- *Lost information cannot be restored.* The `KEYWORD_*` values are absent from v2. No mapping or normalisation
  can reconstruct a distinction the platform no longer reports.
- *Coarsening in one direction is not symmetric.* Harmonising to the common denominator means discarding the finer
  v1 classification for the whole year, so the loss of information is inherited by the v1 data as well.

Both arguments rest on the measurements in 3.1 and 3.2, not on platform documentation.

**Why excluding v2 improves data quality.**
- *One definition per variable.* All rows follow the v1 schema, so columns and category values mean the same thing
  throughout, and no cross-window mapping has to be assumed.
- *Full classification depth.* The analysis keeps `category_specification` (X) and the complete `category` value
  sets of both platforms, which is what the taxonomy comparison of this thesis depends on.
- *No mixing of structural breaks with substantive change.* Any difference observed between platforms or over time
  cannot be an artefact of the schema change.
- *Fewer coverage problems.* The largest gaps in the raw data fall after the break (TikTok 2025-09-05 to
  2025-10-09 and 2025-12-03; X 2025-07-02 to 2025-08-25), as does the year-end effect:
  content posted late in 2025 but reported in 2026 is not part of the download.
- *The window covers the object of study.* The German federal snap election of 2025-02-23 and the weeks around it
  lie fully inside v1.

**Cost of the decision.** The observation period is halved, so no statements about the second half of 2025 and no
before/after comparison across the whole year are possible. This belongs in the limitations section.

**X submission pause within v1.** No X dump between 2025-03-25 and 2025-04-15 contains data. The pause
concerns submission, not moderation: the 16,879 X statements decided on those 22 days were submitted in
the dumps of 2025-04-17 to 2025-05-01, with communication delays of 15 to 23 days, so every decision day
in the window is represented (`03_gold.ipynb`, appendix B).

**Side effect of the content-date rule.** The restriction to content posted in 2025 (section 1)
also removes all 132,130 TikTok rows at `content_date = 2000-01-01`, the schema minimum for that
attribute, which marks them as placeholders rather than posting dates (section 4.3). No separate
placeholder rule is therefore needed.

**Decision.** The main analysis uses the v1 window only: `content_date` from 2025-01-01 to 2025-06-30.

## 4. Cleaning decisions

The decisions applied when the Silver layer is written (section 5). Figures refer to the analysis set
(content posted January to June 2025) unless the full scoped year is stated.

### 4.1 Deduplication

Two identifier attributes were examined for duplicate records. Within the analysis set, the attribute
`uuid` shows 61 excess rows on TikTok (0.0000074%) and none on X; across the full scoped year, 878
`uuid` values occur more than once on TikTok. Groesch et al. (2026) report 12.2 million duplicates
(0.35%) in a corpus of 3.52 billion statements collected between September 2023 and May 2025, so the
present data is substantially cleaner on this identifier. The attribute `platform_uid` shows 302,411
identifiers occurring more than once on TikTok within the analysis set, covering 607,342 rows
(0.0736%), and 335,063 such identifiers across the full scoped year, against 499 comparable cases
reported by Groesch et al. (2026). The Commission's attribute documentation requires that every
platform-specific identifier be unique and not reused, so these records deviate from the reporting
specification.

The duplicated groups were characterised on five dump days spanning January to June 2025. Of 5,061
groups examined, none differed on any substantive attribute: `category`, `decision_visibility`,
`content_type`, `decision_ground`, `content_date`, `application_date`, `automated_detection`,
`automated_decision` and `decision_facts` were identical throughout. Groups differed only in `uuid`
and in `created_at`; on the four days for which the interval was measured, the latter differed by 0
to 115 seconds, with 369 groups exceeding one minute. The pattern is consistent with repeated
submission of the same decision rather than with distinct decisions sharing an identifier.

One row per identifier is retained, selected by the earliest `created_at`. Because the verification
was partitioned by dump day while duplicates could in principle span days, deduplication is applied
across the full period rather than within days. It removes 338,128 TikTok rows across the full scoped
year, of which 304,957 fall within the analysis set; X is unaffected.

### 4.2 Implausible records

905 TikTok rows (0.00011% of the analysis set) report an `application_date` earlier than the
`content_date`, describing a moderation decision applied before the content existed; X contains no
such rows. The condition is logically impossible and indicates a reporting error rather than a
moderation event. Groesch et al. (2026) retained comparable records (0.0006% of their corpus) without
correction. The present analysis removes them, because the German-scoped subset is small enough on the
X side that a record known to be invalid would carry disproportionate weight in per-day comparisons.
Across the full scoped year, the rule removes 2,499 TikTok rows.

Shahi et al. (2025) caution that filtering decisions may disproportionately retain such records,
raising their share within an analysed subset. The scope rule applied here has the opposite effect:
82.2% of Bronze rows survive the scope filter, but only 36.0% of the implausible rows do, which halves
their relative share from 0.000251% to 0.000110%.

### 4.3 Placeholder dates

The Commission's attribute documentation specifies that `content_date` must not precede 2000-01-01 and
`application_date` must not precede 2020-01-01. Values on these bounds are schema minima rather than
plausible dates. The Bronze layer contains 132,130 TikTok rows at `content_date = 2000-01-01` and 11 at
`application_date = 2020-01-01`; X contains none. The restriction to content posted in 2025 removes
these as a side effect; one row on the application-date bound remained in the analysis set before
deduplication. No separate placeholder rule is applied.

The 20 most frequent `content_date` values show no isolated spike that would indicate a further
placeholder convention.

### 4.4 Column handling

All 37 source attributes are retained in the Silver layer. Attribute exclusion is applied at the
analysis layer, where the cost of revising the decision is low; retaining unpopulated attributes in
Silver is inexpensive because parquet encodes them by dictionary.

Six attributes are unpopulated by both platforms across all 1,002,729,268 TikTok and 670,093 X rows:
`account_type`, `decision_ground_reference_url`, `content_language`, `incompatible_content_illegal`,
`decision_monetary_other` and `source_identity`. A comparison against the Commission's attribute
documentation shows that these do not form a single category. Four are optional attributes that both
platforms left blank, which constitutes a reporting choice; for `decision_ground_reference_url` the
documentation explicitly encourages its use. `decision_monetary_other` is required only where
`decision_monetary` takes the value `DECISION_MONETARY_OTHER`, a condition that does not occur in the
data, and `source_identity` is disregarded by the schema where `source_type` is `SOURCE_VOLUNTARY`.
Non-population is therefore a platform choice for four of the six attributes and structurally
determined for the remaining two. The absence of `content_language` corresponds to the observation of
Trujillo et al. (2025) that no social media platform in their sample reported the language of
moderated content.

For `incompatible_content_illegal` the documentation states that leaving the attribute blank does not
imply that the platform considered the content lawful. Non-use is therefore reported without inference
as to its meaning.

`decision_monetary` (11 non-null rows) and `end_date_account_restriction` (6,626 non-null rows) are
not unpopulated and are retained. For the end-date attributes the documentation states that a blank
value denotes an indefinite restriction, so absence carries meaning: against approximately 3.6 million
account terminations, 6,626 specified end dates indicate that account-level restrictions are imposed
indefinitely in nearly all cases.

Attributes populated by one platform and not the other are retained without exception. X populates
`illegal_content_legal_ground` and `illegal_content_explanation` for all rows and does not populate the
`incompatible_content_*` attributes; TikTok shows the inverse. A cross-platform comparison of the
grounds relied on in taking a decision therefore compares two disjoint attributes, which bears directly
on RQ1.

### 4.5 Derived attributes and output

The attributes `decision_visibility`, `content_type`, `category_specification`, `category_addition`
and `territorial_scope` are stored by the database as JSON arrays and are parsed into list types at
this layer. They are not expanded into separate rows: one Silver row corresponds to one statement of
reasons throughout. TikTok submits 9 distinct combinations of `decision_visibility` in the analysis
set, of which 156,759 rows combine age restriction with content removal, so expansion would raise every
subsequent count; across the full scoped year, 295,389 TikTok rows carry more than one value, up to
three. Expansion is applied where required at the analysis layer.

The following attributes are derived: `n_countries_in_scope`, `application_delay` (`application_date`
− `content_date`, in days), `communication_delay` (`created_at` − `application_date`, in days),
`n_filled_optional_fields`, `n_filled_strictly_optional`, `platform` and `scope_class`. Following
Trujillo et al. (2025), the two intervals are reported separately as application delay and
communication delay rather than as a single measure of timeliness.

The count `n_filled_optional_fields` covers the 23 attributes that are not populated in every record:
`decision_visibility`, `decision_visibility_other`, `end_date_visibility_restriction`,
`decision_monetary`, `decision_monetary_other`, `end_date_monetary_restriction`, `decision_provision`,
`end_date_service_restriction`, `decision_account`, `end_date_account_restriction`, `account_type`,
`decision_ground_reference_url`, `illegal_content_legal_ground`, `illegal_content_explanation`,
`incompatible_content_ground`, `incompatible_content_explanation`, `incompatible_content_illegal`,
`category_addition`, `category_specification`, `category_specification_other`, `content_type_other`,
`content_language` and `source_identity`. The remaining 14 attributes are populated in every record
and are excluded: `uuid`, `platform_uid`, `platform_name`, `content_date`, `application_date`,
`created_at`, `territorial_scope`, `decision_ground`, `category`, `content_type`, `decision_facts`,
`automated_detection`, `automated_decision` and `source_type`.

A second count, `n_filled_strictly_optional`, covers 16 of those 23 and excludes the conditionally
required attributes `decision_visibility_other`, `decision_monetary_other`,
`illegal_content_legal_ground`, `illegal_content_explanation`, `incompatible_content_ground`,
`incompatible_content_explanation` and `content_type_other`. Whether these are populated follows from
`decision_ground`, so including them would make X appear more granular merely because it reports
under the illegal-content ground. `category_specification_other` is conditional in the same sense but
is retained in both counts, as the narrower definition excludes only the `decision_*_other` fields and
`content_type_other`; a `content_language_other` attribute does not exist in the schema.

X reports under two platform names, `X` (668,624 rows) and `X (formerly Twitter)` (1,469 rows), counted
in the Bronze layer. A normalised `platform` attribute is derived; the reported value is retained.

The attribute `scope_class` classifies the territorial scope as single country, EU, EEA, or other.
Papaevangelou & Votta (2025) note that the labels EEA and EEA excluding Iceland may denote the same
geographic scope. In the TikTok data the two variants differ only in Iceland: the 29-country list
comprises the EU-27 with Liechtenstein and Norway, and the 30-country list additionally Iceland. The
two variants are classified jointly as EEA while the parsed country list is retained unchanged.
Grouping decisions beyond this classification are taken at the analysis layer.

The Silver layer is written as parquet, partitioned by platform and month of `content_date`. The
partitions are not additionally sorted by `application_date`: because the daily dumps arrive in
submission order, which closely tracks `application_date`, unsorted row groups already span on average
1.1 days of `application_date` (0.4 days when sorted), so statistics-based pruning works nearly as well
without a sort that costs approximately 2.6 hours per partition. The January TikTok partition is sorted
as a by-product of an abandoned sorting pass, while the other eleven are not; sorting changes no values
and has no effect on results. The election period is not applied at this layer: the Silver tables
remain independent of any specific operationalisation so that a revision to the event definition does
not require rebuilding them.

## 5. Write Silver

`build_silver.py` implements section 4 and writes `E:/dsa-data/silver/platform=<P>/month=<M>/`.
The build runs as a script because the TikTok write takes several hours; the cell below only rebuilds
when no Silver files exist or `REBUILD` is set.

- Source: `silver.scoped_*` (Germany in scope, content posted in 2025, whole year).
- Duplicate identifiers are collected in two narrow passes and cached in `E:/dsa-data/silver_meta/`
  (TikTok: 3,090 s for `platform_uid`, 5,458 s for `uuid`). Rows outside duplicate groups are written
  in one scan; only the rows in duplicate groups pass through the window that keeps the earliest
  `created_at`, and are appended in a second statement.
- Measured write time for TikTok: 6,578 s for the non-duplicate rows and 1,734 s for the resolved
  duplicates.
- Two earlier attempts failed by exhausting the scratch directory: a global `ORDER BY application_date`
  over all rows (412.8 GiB), and a query in which one shared subquery was referenced twice, which
  forced full materialisation of all rows (300 GiB). The current design avoids both.

In [8]:
import build_silver

REBUILD = False  # TikTok: ~2.3 h with cached duplicate sets, ~4.6 h without
if REBUILD or not any(SILVER_ROOT.glob("platform=*/month=*/*.parquet")):
    con.close()  # build_silver opens its own connection to the shared database
    for p in PLATFORMS:
        build_silver.build(p)
    con = connect()

SILVER = f"read_parquet('{(SILVER_ROOT / 'platform=*' / 'month=*' / '*.parquet').as_posix()}', hive_partitioning = true)"


### 5.1 Cleaning funnel

Rows per platform at each step. Deduplication removals follow as the remainder, since implausible rows
are removed before duplicates are resolved.

In [9]:
funnel = []
for p in PLATFORMS:
    bronze = con.sql(f"SELECT count(*) FROM bronze.{p}").fetchone()[0]
    scoped, implausible = con.sql(f"""
        SELECT count(*), count(*) FILTER (WHERE application_date < content_date)
        FROM silver.scoped_{p}""").fetchone()
    silver = con.sql(f"SELECT count(*) FROM {SILVER} WHERE platform = '{PLATFORM_NAMES[p]}'").fetchone()[0]
    funnel.append({"platform": p, "bronze": bronze, "scoped": scoped,
                   "implausible_removed": implausible,
                   "dedup_removed": scoped - implausible - silver, "silver": silver})

pd.DataFrame(funnel).set_index("platform")

,bronze,scoped,implausible_removed,dedup_removed,silver
platform,,,,,
tiktok,1002729268,979896444,2499,338128,979555817
x,670093,183321,0,0,183321


### 5.2 One row per statement, and distributions against the pre-cleaning figures

List attributes keep several values per row, and the distributions within the analysis set match the
pre-cleaning figures of section 3 apart from the removed rows.

In [10]:
print(con.sql(f"""
    SELECT platform, count(*) AS rows_total,
           count(*) FILTER (WHERE content_date < DATE '{SCHEMA_BREAK}') AS rows_analysis_set,
           max(len(decision_visibility)) AS max_visibility_values,
           count(*) FILTER (WHERE len(decision_visibility) > 1) AS rows_with_several_values
    FROM {SILVER} GROUP BY platform ORDER BY platform""").df().to_string(index=False))

combos = con.sql(f"""
    SELECT platform, decision_visibility, content_type, count(*) AS n
    FROM {SILVER} WHERE content_date < DATE '{SCHEMA_BREAK}'
    GROUP BY ALL""").df()

for name, col in (("TikTok", "content_type"), ("X", "decision_visibility")):
    d = combos[combos["platform"] == name]
    total = d["n"].sum()
    shares = (d.explode(col).groupby(col)["n"].sum().sort_values(ascending=False)
                .to_frame().assign(pct_of_rows=lambda x: (100 * x["n"] / total).round(3)))
    print(f"\n{name} {col} (row counted once per value, {total:,} rows)")
    print(shares.to_string())

tk = combos[combos["platform"] == "TikTok"]
vis = tk.groupby(tk["decision_visibility"].map(lambda v: tuple(v) if hasattr(v, "__len__") else None),
                 dropna=True)["n"].sum()
both = ("DECISION_VISIBILITY_CONTENT_AGE_RESTRICTED", "DECISION_VISIBILITY_CONTENT_REMOVED")
print(f"\nTikTok decision_visibility: {len(vis)} distinct combinations; "
      f"age restriction + removal: {int(vis.get(both, 0)):,} rows")

platform  rows_total  rows_analysis_set  max_visibility_values  rows_with_several_values
  TikTok   979555817          824228641                      3                    295389
       X      183321             132191                      1                         0



TikTok content_type (row counted once per value, 824,228,641 rows)
                              n  pct_of_rows
content_type                                
CONTENT_TYPE_TEXT     453954523       55.076
CONTENT_TYPE_VIDEO    286788949       34.795
CONTENT_TYPE_IMAGE     66123819        8.023
CONTENT_TYPE_OTHER     16636429        2.018
CONTENT_TYPE_AUDIO       741136        0.090
CONTENT_TYPE_PRODUCT       3995        0.000

X decision_visibility (row counted once per value, 132,191 rows)
                                n  pct_of_rows
decision_visibility                           
DECISION_VISIBILITY_OTHER  132191        100.0

TikTok decision_visibility: 9 distinct combinations; age restriction + removal: 156,759 rows


### 5.3 Size on disk and free-text share

In [11]:
import os

for name in PLATFORM_NAMES.values():
    files = list((SILVER_ROOT / f"platform={name}").rglob("*.parquet"))
    cols = con.sql(f"""
        SELECT path_in_schema AS col, sum(total_compressed_size) AS bytes
        FROM parquet_metadata('{(SILVER_ROOT / f'platform={name}' / 'month=*' / '*.parquet').as_posix()}')
        GROUP BY 1""").df()
    text = cols[cols["col"].isin(["decision_facts", "illegal_content_explanation",
                                  "incompatible_content_explanation"])]["bytes"].sum()
    print(f"{name}: {sum(os.path.getsize(f) for f in files) / 1e9:.2f} GB on disk, {len(files)} files, "
          f"free-text columns {100 * text / cols['bytes'].sum():.1f}% of column data")
    print(cols.sort_values("bytes", ascending=False).head(4)
              .assign(gb=lambda d: (d["bytes"] / 1e9).round(3))[["col", "gb"]].to_string(index=False))

TikTok: 28.71 GB on disk, 23 files, free-text columns 2.7% of column data
                             col     gb
                            uuid 20.175
                    platform_uid  5.508
incompatible_content_explanation  0.744
     incompatible_content_ground  0.465
X: 0.01 GB on disk, 12 files, free-text columns 2.1% of column data
                         col    gb
                        uuid 0.004
                platform_uid 0.001
                  created_at 0.000
illegal_content_legal_ground 0.000


### 5.4 Query time: Silver parquet versus the Bronze-based views

The same aggregate, rows per platform and month, against the written Silver parquet and against the
views over Bronze that it replaces.

In [12]:
import time

t = time.time()
con.sql(f"SELECT platform, month, count(*) FROM {SILVER} GROUP BY ALL").fetchall()
t_silver = time.time() - t

t = time.time()
for p in PLATFORMS:
    con.sql(f"SELECT date_trunc('month', content_date), count(*) FROM silver.scoped_{p} GROUP BY 1").fetchall()
t_views = time.time() - t

print(f"Silver parquet: {t_silver:.1f} s | Bronze-based views: {t_views:.1f} s | "
      f"speed-up: {t_views / t_silver:.0f}x")

Silver parquet: 1.4 s | Bronze-based views: 26.4 s | speed-up: 19x


## 6. Analysis layer: v1 window

Following the decision in 3.3, the analysis dataset is the v1 window only, now read from the cleaned
Silver parquet:

| Rule | Column |
|---|---|
| Germany is in the territorial scope | `territorial_scope` contains `"DE"` |
| Content posted in 2025 | `year(content_date) = 2025` |
| Before the schema break | `content_date < 2025-07-01` |

`silver.sor` exposes the whole Silver layer; `silver.analysis_*` restricts it to the v1 window and is
the dataset all further analysis builds on. `silver.scoped_*` (Bronze-based, whole year, before
cleaning) is kept, because sections 2 and 3 compare the two windows on it.

In [13]:
con.execute(f"CREATE OR REPLACE VIEW silver.sor AS SELECT * FROM {SILVER}")
for p in PLATFORMS:
    con.execute(f"""
        CREATE OR REPLACE VIEW silver.analysis_{p} AS
        SELECT * FROM silver.sor
        WHERE platform = '{PLATFORM_NAMES[p]}' AND content_date < DATE '{SCHEMA_BREAK}'
    """)

pd.DataFrame([
    {"platform": p,
     **dict(zip(["rows", "first_content_date", "last_content_date"],
                con.sql(f"SELECT count(*), min(content_date), max(content_date) "
                        f"FROM silver.analysis_{p}").fetchone()))}
    for p in PLATFORMS
])

,platform,rows,first_content_date,last_content_date
0,tiktok,824228641,2025-01-01,2025-06-30
1,x,132191,2025-01-01,2025-06-30
